# Bike Rebalancing ML — Google Colab Runner

Runs the full pipeline: **weather → preprocess → features → train → evaluate → rebalance → visualize**.

### Before you start
1. Upload `bike_pipeline.zip` (~270 MB: code + **already-cleaned** 24-month parquet data). Uploading it to Google Drive first and copying from there is much faster than the browser upload.
2. The raw year ZIPs are **not needed** — preprocessing was already run and validated locally. Only upload them if you want to re-run the preprocess stage.
3. (Optional) Runtime → Change runtime type → **High-RAM** and/or **GPU** for faster training.
4. Then: Runtime → **Run all**.

## 1. Get the project code

**Option A — upload the zip** (run this cell, then pick `bike_pipeline.zip`):

In [ ]:
# Finds bike_pipeline*.zip on Google Drive (fast) or asks you to upload it,
# extracts it to /content/prj and checks that the project files are there.
import os, glob, shutil, zipfile
from google.colab import drive, files

drive.mount('/content/drive')
os.chdir('/content')
MIN_MB = 100  # the real bundle is ~276 MB; anything tiny is an old/broken copy

def size_mb(p): return os.path.getsize(p) / 1e6

cands = (glob.glob('/content/drive/MyDrive/bike_pipeline*.zip')
         + glob.glob('/content/drive/MyDrive/*/bike_pipeline*.zip')
         + glob.glob('/content/bike_pipeline*.zip'))
for c in cands: print(f'found {c}  ({size_mb(c):.0f} MB)')
good = [c for c in cands if size_mb(c) >= MIN_MB]

if good:
    src = max(good, key=os.path.getmtime)
else:
    print('No valid bike_pipeline.zip on Drive — choose it from E:\\prj to upload ...')
    up = files.upload()
    src = os.path.join('/content', max(up, key=lambda k: len(up[k])))
print(f'Using {src} ({size_mb(src):.0f} MB)')
if size_mb(src) < MIN_MB:
    raise RuntimeError('Zip is too small — the Drive upload may be incomplete or an old copy. Re-upload E:\\prj\\bike_pipeline.zip (276 MB).')

shutil.rmtree('/content/prj', ignore_errors=True)
with zipfile.ZipFile(src) as z:
    z.extractall('/content/prj')

# If the zip had an extra top-level folder, move its contents up
if not os.path.exists('/content/prj/main.py'):
    hits = glob.glob('/content/prj/**/main.py', recursive=True)
    if hits:
        inner = os.path.dirname(hits[0])
        for name in os.listdir(inner):
            shutil.move(os.path.join(inner, name), '/content/prj')

missing = [f for f in ['main.py', 'requirements.txt', 'colab_patch.py', 'configs/config_colab.yaml']
           if not os.path.exists(os.path.join('/content/prj', f))]
if missing:
    raise RuntimeError(f'Extracted zip is missing {missing} — wrong zip?')
n_months = len(glob.glob('/content/prj/data/processed/*_station_time.parquet'))
print(f'OK: project extracted, {n_months} month files in data/processed')
%cd /content/prj

**Option B — clone from GitHub** (edit the URL, then run instead of Option A):

In [ ]:
# !git clone https://github.com/<you>/<repo>.git /content/prj
# %cd /content/prj
# !ls -la

## 2. (Optional) Raw data ZIPs — only needed to re-run preprocessing

Skip this. The cleaned parquet files are already in `data/processed/`.

In [ ]:
# Uncomment to copy the raw ZIPs from Drive into the project folder
# !cp '/content/drive/MyDrive/citibike/2022-citibike-tripdata.zip' /content/prj/
# !cp '/content/drive/MyDrive/citibike/2023-citibike-tripdata.zip' /content/prj/

## 3. Install dependencies

In [ ]:
%cd /content/prj
!pip install -q -r /content/prj/requirements.txt

## 4. Verify the cleaned data arrived intact

Should end with `PROBLEMS: none` (24 files, complete 10-min grids, no nulls/overlaps).

In [ ]:
!python _audit_parquet.py | grep -E 'files|PROBLEMS|station_master'

## 5. Run the pipeline

Uses `configs/config_colab.yaml`: **all 24 months**, hourly predictions per station, weather features.

- **features**: builds one feature file per month (~3 min)
- **train**: Baseline, Random Forest, XGBoost for pickups & dropoffs, an XGBoost net-flow (surplus/deficit) model, and a no-weather XGBoost to measure weather impact. Trains on a sample spread across every training month (`max_train_rows`; lower it to 3000000 if a standard runtime runs out of RAM)
- **evaluate**: full test period (last ~3.5 months of 2023)
- **rebalance**: plans truck moves for each morning/evening peak and scores them against what actually happened

The **preprocess** stage is skipped by default because the data is already cleaned.

In [ ]:
CONFIG = 'configs/config_colab.yaml'

In [ ]:
!python main.py --config {CONFIG} --stage weather

In [ ]:
# Already done locally. Uncomment only if you uploaded the raw ZIPs (step 2).
# !python main.py --config {CONFIG} --stage preprocess

In [ ]:
!python main.py --config {CONFIG} --stage features

In [ ]:
!python main.py --config {CONFIG} --stage train

In [ ]:
!python main.py --config {CONFIG} --stage evaluate

In [ ]:
!python main.py --config {CONFIG} --stage rebalance

In [ ]:
!python main.py --config {CONFIG} --stage visualize

## 6. View results

In [ ]:
import json, glob
from IPython.display import Image, display

ev = json.load(open('outputs/metrics/evaluation_report.json'))
tr = json.load(open('outputs/metrics/training_report.json'))
rb = json.load(open('outputs/metrics/rebalancing_report.json'))
print('Split:', ev['split_dates'])
print('\nTEST MAE (bikes per station-hour, lower is better)')
for tgt, models in ev['test_metrics'].items():
    for name, m in models.items():
        if not name.startswith('_'):
            print(f"  {tgt:8s} {name:24s} MAE={m['mae']:.3f}  R2={m['r2']}" + ('  <= best' if name == ev['best_model'][tgt] else ''))
print('\nWeather impact (validation):', json.dumps(tr['weather_impact'], indent=1))
print('\nSurplus/deficit, 3-hour blocks:')
for k, v in ev['net_flow_evaluation'].items():
    if isinstance(v, dict) and 'net_flow_mae' in v:
        print(f"  {k:24s} net MAE={v['net_flow_mae']:.2f}  direction acc={v['direction_accuracy_material']:.1%}")
print('\nRebalancing (share of perfect-forecast benefit):')
for k, v in rb['summary'].items():
    print(f"  {k:12s} shortage -{v['shortage_reduction_pct']}%   {v['pct_of_perfect_forecast_benefit']}% of ideal")

# Show the PNG figures
for png in sorted(glob.glob('outputs/figures/*.png')):
    print(png)
    display(Image(png))

In [ ]:
# Interactive map: predicted surplus/deficit + truck moves for a sample peak window
from IPython.display import HTML
HTML(open('outputs/figures/12_station_map.html').read())

## 7. (Optional) Save outputs back to Drive

So the processed parquet, trained models, and figures survive the runtime disconnect.

In [ ]:
!mkdir -p '/content/drive/MyDrive/citibike_outputs'
!cp -r /content/prj/outputs '/content/drive/MyDrive/citibike_outputs/'
!cp -r /content/prj/models  '/content/drive/MyDrive/citibike_outputs/'
print('Saved to MyDrive/citibike_outputs')